# 07 — Scaling up: the head on 800 records

**Where we are.** Notebook 6 trained the head on 8 records and learned two things: a frozen backbone with a small head memorises easily, and 8 records are far too few to say whether it generalises. Here we use `data/synthetic/`: 3,000 support tickets (2,400 train, 300 val, 300 test) generated by any2jev's synthetic generator and converted to our schema by `jev/data.py`.

**What's new here:**
1. **Batched, padded feature extraction.** The backbone runs on 64 questions at a time, and the readout vectors go straight into a few big GPU tensors (`jev/features.py`). A training batch is then just an index slice, with no Python loop over examples.
2. **A masked loss.** Questions have different numbers of options, so score tensors are padded, and the padding must not take part in the softmax. You'll write this one.
3. **A learning-rate schedule.** Notebook 6 hinted that learning rates matter. At this scale they decide whether training is stable.
4. **Discipline about data.** We tune on the validation set and touch the test set exactly once, at the end.
5. **Experiments instead of opinions.** A learning curve, a shuffling ablation, and a look at the errors.

**A caveat that shapes how to read every number below:** this data is synthetic and template-based, and the label is encoded in keywords ("URGENT", "furious", "refund"). High accuracy here shows the pipeline works. It does not show the model would handle real tickets.

The new code is in `jev/head.py` (a batch-first `MaskedPointerHead`, with optional input normalisation) and `jev/features.py` (`build_requests`, `extract`, `FeatureSet`). Your notebook 5 and 6 code is untouched. Read the two files; they are short.

> **GPU memory:** shut down the kernel from notebook 6 first.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import sys
sys.path.insert(0, "..")

import json, math, time, statistics
from collections import defaultdict

import torch
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForMultimodalLM

from jev.render import Renderer
from jev.head import MaskedPointerHead
from jev.data import load_jsonl
from jev.features import build_requests, extract

MODEL_ID = "Qwen/Qwen3.5-0.8B"
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto").eval()

core = model.model
for p in core.parameters():
    p.requires_grad_(False)

renderer = Renderer(tokenizer)
device = next(core.parameters()).device
HIDDEN = model.config.text_config.hidden_size
PAD = tokenizer.pad_token_id
print("loaded | GPU memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))

## Step 1: the data

Look at the converted records before training on them. Two things from the converter to notice: option text sometimes includes a description (`"billing: Charges, invoices, ..."`), and the state can be a flattened dict or chat.

In [ ]:
train_records = load_jsonl("../data/synthetic/train.jsonl")
val_records = load_jsonl("../data/synthetic/val.jsonl")
print(len(train_records), "train records,", len(val_records), "val records (the 300 test records stay untouched until the end)\n")

rec = train_records[3]
print("STATE:\n" + rec["state"] + "\n")
for q in rec["questions"]:
    print(f"[{q['type']}] {q['id']}: {q['prompt']}")
    for i, o in enumerate(q["options"]):
        print(f"   {'*' if i == q['label'] else ' '} {i}: {o}")

## Step 2: cache the backbone features, in batches

Same idea as notebook 6 (the backbone is frozen, so run it once), but now it has to be fast enough for 14,000+ examples. Three changes:
- **Batches of 64, sorted by length**, so padding is small.
- **Right padding with no special handling.** The backbone is causal, so a real token never sees the padding that comes after it, and the readout vectors are the same as without padding. You can verify that below.
- **Pre-padded storage.** The vectors for all examples go into three tensors (`opt [E, N_max, D]`, `dec [E, D]`, `mask [E, N_max]`) in bf16 on the GPU. bf16 is lossless here because the backbone's output is already bf16.

First, check that batching doesn't change the answer. Do you expect the vectors to be *identical* to the one-at-a-time ones, or just very close?

In [ ]:
probe = build_requests(train_records[:10], n_orders=2, seed=0)[:24]
batched = extract(core, renderer, probe, PAD, HIDDEN, batch_size=8)
single = extract(core, renderer, probe, PAD, HIDDEN, batch_size=1)

cos_dec = F.cosine_similarity(batched.dec.float(), single.dec.float(), dim=-1)
cos_opt = F.cosine_similarity(batched.opt.float(), single.opt.float(), dim=-1)[batched.mask]
print("decide vectors: min cosine %.5f | option vectors: min cosine %.5f" % (cos_dec.min().item(), cos_opt.min().item()))
print("bit-identical rows: %d of %d" % ((batched.dec == single.dec).all(-1).sum().item(), len(batched)))
print("padding slots are exactly zero:", bool((batched.opt[~batched.mask] == 0).all()))

**Why close but not identical?** A bf16 matrix multiply rounds differently depending on the shapes involved, so a batch of 8 and a batch of 1 give slightly different low-order bits. This is the same effect that made notebook 3's cache check look alarming until we compared against fp32. A cosine above 0.999 means the vectors point the same way to within rounding.

Now cache the real sets. Each training question is shown in 6 option orders (the original plus 5 shuffles with the label moved); validation uses 4 orders so we can measure order stability. Raise `N_TRAIN` to use more data.

In [ ]:
N_TRAIN, N_VAL = 800, 300

torch.cuda.synchronize(); t0 = time.time()
train_fs = extract(core, renderer, build_requests(train_records[:N_TRAIN], n_orders=6, seed=1), PAD, HIDDEN)
torch.cuda.synchronize(); t_train = time.time() - t0

t0 = time.time()
val_fs = extract(core, renderer, build_requests(val_records[:N_VAL], n_orders=4, seed=2), PAD, HIDDEN)
torch.cuda.synchronize(); t_val = time.time() - t0

print(f"train: {len(train_fs)} examples in {t_train:.0f}s ({1000 * t_train / len(train_fs):.1f} ms each)")
print(f"val:   {len(val_fs)} examples in {t_val:.0f}s")
print("max options per question:", train_fs.opt.shape[1])
print("cached features: %.0f MB" % ((train_fs.opt.numel() + train_fs.dec.numel() + val_fs.opt.numel() + val_fs.dec.numel()) * 2 / 1e6))

## Step 3: the masked loss

The head returns a score tensor of shape `[batch, N_max]`, where `N_max` is 5 here. A question with 3 options has 3 real scores and 2 padded slots, and those padded slots hold arbitrary numbers. If they enter the softmax, they steal probability from the real options and the loss is wrong.

Write `masked_cross_entropy(scores, mask, labels)`: the mean cross-entropy over the batch, where only the real options take part. Remember what `exp(-inf)` is. The cell after it checks your answer against the slow, obviously-correct way (one example at a time), and also shows what happens if you forget the mask.

In [ ]:
def masked_cross_entropy(scores, mask, labels):
    """scores [B, N] (junk in padded slots), mask [B, N] (True = real option), labels [B]  ->  scalar mean loss"""
    # TODO
    raise NotImplementedError("fill in masked_cross_entropy")

<details><summary>Solution (open after you've tried)</summary>

```python
return F.cross_entropy(scores.masked_fill(~mask, float("-inf")), labels)
```

A score of `-inf` becomes `exp(-inf) = 0` in the softmax, so that slot gets zero probability and the remaining options renormalise among themselves. The label is never a padded slot, so the loss is always finite.

</details>

In [ ]:
head_check = MaskedPointerHead(HIDDEN).to(device)
o, d, m, l = train_fs.batch(torch.arange(32, device=device))
scores = head_check(o, d)     # no mask passed, so padded slots hold ordinary-looking numbers

fast = masked_cross_entropy(scores, m, l)
slow = torch.stack([F.cross_entropy(scores[i, m[i]].unsqueeze(0), l[i:i + 1]) for i in range(len(l))]).mean()
forgot = F.cross_entropy(scores, l)

print("masked batch loss     : %.6f" % fast.item())
print("one example at a time : %.6f" % slow.item())
print("forgot the mask       : %.6f   <- silently wrong" % forgot.item())
assert torch.allclose(fast, slow, atol=1e-5), "masked loss does not match the one-at-a-time loss"

## Step 4: training, with a learning-rate schedule

`train` below is complete (read it). Three choices to understand:
- **Batches are index slices** of the cached tensors, so an epoch over 14,520 examples takes about a second.
- **A one-cycle schedule**: the learning rate climbs from near zero to `lr` over the first 10% of steps, then decays along a cosine curve to near zero. Notebook 6 used a constant rate.
- **`normalize=True` in the head** applies an RMS normalisation to the backbone vectors before the projections. Their norm is around 110, and notebook 6 showed what that does to training at higher learning rates.

In [ ]:
@torch.no_grad()
def evaluate(head, fs):
    head.eval()
    preds, loss_sum = [], 0.0
    for i in range(0, len(fs), 512):
        o, d, m, l = fs.batch(torch.arange(i, min(i + 512, len(fs)), device=device))
        sc = head(o, d, m)
        preds.append(sc.argmax(-1))
        loss_sum += F.cross_entropy(sc, l, reduction="sum").item()
    preds = torch.cat(preds)
    correct = (preds == fs.labels).cpu().tolist()

    by_q, picked = defaultdict(list), defaultdict(set)
    for ok, p, qid, rec, typ, opts in zip(correct, preds.tolist(), fs.meta["qid"], fs.meta["rec"], fs.meta["type"], fs.meta["options"]):
        by_q[qid].append(ok)
        if typ != "score":
            picked[(rec, qid)].add(opts[p])
    return {
        "acc": sum(correct) / len(correct),
        "loss": loss_sum / len(fs),
        "by_question": {k: sum(v) / len(v) for k, v in by_q.items()},
        "stability": sum(len(s) == 1 for s in picked.values()) / len(picked),
    }


def train(fs, val_fs, epochs=30, lr=1e-3, batch_size=64, weight_decay=0.01, seed=0, normalize=True, schedule=True, log_every=0):
    torch.manual_seed(seed)
    head = MaskedPointerHead(HIDDEN, normalize=normalize).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    steps = epochs * math.ceil(len(fs) / batch_size)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=steps, pct_start=0.1, anneal_strategy="cos") if schedule else None
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)
    history = []
    for epoch in range(epochs):
        head.train()
        total = 0.0
        for o, d, m, l in fs.epoch(batch_size, generator):
            optimizer.zero_grad()
            loss = masked_cross_entropy(head(o, d), m, l)
            loss.backward()
            optimizer.step()
            if scheduler:
                scheduler.step()
            total += loss.item() * len(l)
        v = evaluate(head, val_fs)
        history.append({"train_loss": total / len(fs), "val_loss": v["loss"], "val_acc": v["acc"]})
        if log_every and (epoch % log_every == 0 or epoch == epochs - 1):
            print(f"epoch {epoch:2d}  train loss {history[-1]['train_loss']:.4f} | val loss {v['loss']:.4f} | val acc {v['acc']:.3f}")
    return head, history

### The first real run

Chance accuracy is what you get by guessing uniformly among the options. It's the number to beat.

In [ ]:
def chance_accuracy(fs):
    return sum(1 / len(o) for o in fs.meta["options"]) / len(fs)

print("chance accuracy on val: %.3f\n" % chance_accuracy(val_fs))

t0 = time.time()
head, history = train(train_fs, val_fs, epochs=30, lr=1e-3, log_every=3)
print("\ntraining took %.1fs" % (time.time() - t0))

r = evaluate(head, val_fs)
print(f"\nval: accuracy {r['acc']:.3f} | loss {r['loss']:.3f} | order stability {r['stability']:.3f}")
print("by question:", {k: round(v, 3) for k, v in r["by_question"].items()})

**Questions (answer in a comment before opening mine):**
1. Compare the train and val losses across epochs. Is there a gap, and does it look like notebook 6's overfitting (train accuracy 1.0, val near chance)? What changed?
2. Which question is hardest, and which easiest? Does that match what you'd guess from how the generator builds the tickets?
3. "Order stability" skips `score` questions. Why?

<details><summary>My answers</summary>

1. There is a gap in the loss (train ends near 0.003 against a val loss of about 0.13), so the head still memorises the training tickets, and the val loss stops improving after about epoch 20. But unlike notebook 6, val accuracy is high (about 0.96 against 0.35 for chance) and rose together with the training accuracy. The difference is data: 800 records is 100 times more distinct text than 8, so what the head learns now transfers to tickets it has not seen.
2. `tone` and `urgency` are easiest (about 0.998 and 0.996 on my run), `refund` next (0.98), and `department` is far the hardest (0.86): it has more options, some tickets are genuinely ambiguous, and some have the true department removed. That `urgency` is nearly solved is the keyword caveat in action: the generator expresses urgency with a handful of fixed phrases ("URGENT", "no rush"), so a 3-level ordinal scale is easy here and would not be on real text.
3. `score` options have a fixed order by design (calm < frustrated < furious), so we never shuffle them, and "does the answer survive reordering" has no meaning for them.

</details>

## Experiment A: how much does the learning-rate schedule matter?

Same data, same head, five seeds each. We compare a constant learning rate (notebook 6's choice) with the one-cycle schedule, and also check whether the input normalisation earns its place.

Predict first: which settings will have the largest spread across seeds?

In [ ]:
configs = [
    ("constant lr 1e-3, 12 epochs",           dict(epochs=12, schedule=False)),
    ("one-cycle 1e-3, 30 epochs",             dict(epochs=30, schedule=True)),
    ("one-cycle 1e-3, 30 epochs, no norm",    dict(epochs=30, schedule=True, normalize=False)),
]
for name, kw in configs:
    accs, losses = [], []
    for seed in range(5):
        h, _ = train(train_fs, val_fs, seed=seed, lr=1e-3, **kw)
        r = evaluate(h, val_fs)
        accs.append(r["acc"]); losses.append(r["loss"])
    print(f"{name:38s} val acc {statistics.mean(accs):.3f} +- {statistics.pstdev(accs):.3f} (worst {min(accs):.3f}) | val loss {statistics.mean(losses):.3f}")

<details><summary>What I saw</summary>

Constant `lr=1e-3`: val accuracy **0.921 +- 0.038** across seeds (worst seed 0.847), val loss 0.332. One-cycle schedule: **0.960 +- 0.001** (worst 0.959), val loss 0.126. So the schedule removes the seed-to-seed noise and lifts the average by four points; the same peak learning rate that makes a constant rate bounce around is fine once it decays. Dropping the input normalisation leaves accuracy unchanged (0.959 +- 0.001) but the val loss rises to 0.208 from 0.126, meaning its probabilities are worse at the same accuracy. That will matter for calibration in the next notebook. Notebook 6's lesson holds: the learning rate matters, and a schedule is the fix.

</details>

## Experiment B: the learning curve

Notebook 6 suggested that more data helps but could not show it cleanly. Now we can: train on the first 25, 50, 100, ... 800 records (using the same cached features) and watch validation accuracy.

In [ ]:
print("records  examples   val acc            order stability")
for k in [25, 50, 100, 200, 400, 800]:
    sub = train_fs.where_rec_below(k)
    accs, stabs = [], []
    for seed in range(3):
        h, _ = train(sub, val_fs, epochs=15, seed=seed)
        r = evaluate(h, val_fs)
        accs.append(r["acc"]); stabs.append(r["stability"])
    print(f"{k:6d}  {len(sub):8d}   {statistics.mean(accs):.3f} +- {statistics.pstdev(accs):.3f}      {statistics.mean(stabs):.3f}")

<details><summary>What I saw</summary>

Validation accuracy rose with every doubling of the data: 0.563 (25 records), 0.656 (50), 0.761 (100), 0.835 (200), 0.919 (400), 0.949 (800), and order stability went from 0.65 to 0.89. The gain per doubling is 0.07 to 0.10 up to 400 records and 0.03 for the last one, so it is slowing but has not flattened, and we have 2,400 training records available. The spread across seeds is under 0.01 throughout, so these are real differences. This confirms notebook 6's suspicion: the weak held-out numbers there came from having 8 records. One caution: this text has only 8 phrases per department, so the curve will flatten sooner than it would on real tickets, where the variety is unbounded.

</details>

## Experiment C: does shuffling the options still pay off?

Notebook 6 found that shuffling raised order stability on 12 questions, with the caveat that 12 questions is thin. Repeat it with 800 records. The baseline sees only the original option order, and gets 6 times more epochs so both runs take about the same number of gradient steps.

In [ ]:
train_fs_1order = extract(core, renderer, build_requests(train_records[:N_TRAIN], n_orders=1, seed=1), PAD, HIDDEN)

for name, fs, epochs in [("1 order (original only)", train_fs_1order, 90), ("6 orders (shuffled)", train_fs, 15)]:
    accs, stabs = [], []
    for seed in range(3):
        h, _ = train(fs, val_fs, epochs=epochs, seed=seed)
        r = evaluate(h, val_fs)
        accs.append(r["acc"]); stabs.append(r["stability"])
    print(f"{name:26s} val acc {statistics.mean(accs):.3f} | order stability {statistics.mean(stabs):.3f}")

<details><summary>What I saw</summary>

One order: val accuracy 0.887 and order stability 0.726. Six orders: 0.949 and 0.892. Shuffling is worth about six points of accuracy and seventeen points of stability, with no downside. That is a much clearer result than the 12-question version in notebook 6 could give. Part of the accuracy gain is that six orderings of each question also act as data augmentation.

</details>

## Error analysis: what does it get wrong?

Accuracy is one number; the mistakes tell you what to fix. Start with the error rate per true answer on the `department` question. The generator sometimes removes the true department from the options and adds a "none of the above" option (under the keys `other`, `none` or `not_listed`) as the *correct* answer, so those are listed too.

In [ ]:
o, d, m, l = val_fs.batch(torch.arange(len(val_fs), device=device))
with torch.no_grad():
    pred = head(o, d, m).argmax(-1).tolist()

total, wrong = defaultdict(int), defaultdict(lambda: defaultdict(int))
for i, p in enumerate(pred):
    if val_fs.meta["qid"][i] != "department":
        continue
    opts = val_fs.meta["options"][i]
    true, got = opts[val_fs.labels[i].item()].split(": ")[0], opts[p].split(": ")[0]
    total[true] += 1
    if true != got:
        wrong[true][got] += 1

print(f"{'true answer':13s} {'n':>4s}  {'error rate':>10s}   most common mistakes")
for true in sorted(total, key=lambda t: -sum(wrong[t].values()) / total[t]):
    n_wrong = sum(wrong[true].values())
    top = sorted(wrong[true].items(), key=lambda kv: -kv[1])[:3]
    print(f"{true:13s} {total[true]:4d}  {n_wrong / total[true]:10.1%}   {top}")

Careful with that table: `n` counts *examples*, and every ticket appears in 4 option orders, so one badly handled ticket contributes up to 4 errors. Look at the errors per distinct ticket instead.

In [ ]:
by_ticket = defaultdict(lambda: [0, 0])      # record index -> [wrong orders, total orders]
for i, p in enumerate(pred):
    if val_fs.meta["qid"][i] == "department":
        k = val_fs.meta["rec"][i]
        by_ticket[k][1] += 1
        by_ticket[k][0] += p != val_fs.labels[i].item()

bad = [(k, w, t) for k, (w, t) in by_ticket.items() if w > 0]
print(f"{len(bad)} of {len(by_ticket)} tickets have at least one wrong option order; {sum(w == t for _, w, t in bad)} are wrong in every order\n")
for k, w, t in sorted(bad, key=lambda x: -x[1])[:10]:
    q = next(q for q in val_records[k]["questions"] if q["id"] == "department")
    true = q["options"][q["label"]].split(": ")[0]
    options = [o.split(": ")[0] for o in q["options"]]
    print(f"wrong in {w}/{t} orders | true={true:9s} | options={options}\n    {val_records[k]['state'][:110]!r}")

**Questions:**
1. Which true answers have the highest error rates, and is "none of the above" harder than a real department?
2. Are the errors spread evenly over many tickets, or concentrated? Read the ticket texts: are the mistakes foolish, or are the labels debatable?
3. For `true=other`, what does the model do instead?
4. Which of these would you attack with more data, and which needs something different?

<details><summary>My answers</summary>

1. On my run: `returns` 22% error (mistaken for shipping 18 times and technical 17), `other` 21%, billing 12%, shipping 11%, `not_listed` 10%, technical 10%, `none` 5%. So "none of the above" is not uniformly harder: only the bare `other` option is, and the other two are no worse than real departments.
2. Both kinds. 66 of 235 tickets have at least one wrong option order, but only 4 are wrong in every order. So much of the error is order sensitivity: the same ticket is right in some orders and wrong in others, which is what the order stability of about 0.89 measures. The tickets that are wrong consistently repeat a few phrases. "The screen was cracked when I opened it" is labelled `returns`, but `technical` is not a foolish answer, and that phrase alone accounts for 4 of the 10 worst tickets. The generator also adds a random refund sentence, and it may be what pulls "Two-factor codes never arrive. I would like a refund." (true answer `technical`) away from the right department.
3. When the generator deletes the true department, the model picks the nearest remaining option instead of `other`: a parcel left at the wrong address, with `shipping` missing from the options, went to `returns`. That is a real weakness, since recognising that *nothing* matches is harder than matching.
4. The debatable labels are not a data problem; they set a ceiling for this generator. The order sensitivity is what shuffling already attacks, and more orders or more data should reduce it further. The missing-answer case could improve with more examples of it, or with a backbone that can adapt (LoRA), since the frozen vectors may carry little 'nothing matches' signal. That is a question for the LoRA notebook.

</details>

## The test set, once

We have tuned the schedule, the normalisation and the epoch count against the validation set. The test set has not been used for any decision, so it gives an unbiased estimate. We run it once and report whatever it says. If you tweak anything after looking at this number, it stops being a test set.

In [ ]:
test_records = load_jsonl("../data/synthetic/test.jsonl")
test_fs = extract(core, renderer, build_requests(test_records, n_orders=4, seed=3), PAD, HIDDEN)

final_head, _ = train(train_fs, val_fs, epochs=30, lr=1e-3, seed=0)
for name, fs in [("val", val_fs), ("test", test_fs)]:
    r = evaluate(final_head, fs)
    print(f"{name:5s} accuracy {r['acc']:.3f} | loss {r['loss']:.3f} | order stability {r['stability']:.3f} | chance {chance_accuracy(fs):.3f}")
    print("      by question:", {k: round(v, 3) for k, v in r["by_question"].items()})

## Wrap-up

What we found, roughly in order of how much to trust it:

1. **Scale fixed the generalisation problem from notebook 6.** With 800 records validation accuracy is about 0.96 against 0.35 for chance, and the test set, used once, agrees (0.959, order stability 0.90). The close match between validation and test means we did not overfit our choices to the validation set.
2. **The learning-rate schedule is worth four points and removes seed noise.** A constant rate gave 0.921 +- 0.038 and the schedule 0.960 +- 0.001.
3. **Shuffling options is worth six points of accuracy and seventeen of order stability.**
4. **We are data-limited, not capacity-limited.** The learning curve is still rising at 800 records.
5. **The remaining errors are mostly debatable labels, plus a genuine weakness on "none of the above".**

The caveat from the start still applies: this is synthetic, keyword-driven text. Treat 0.96 as evidence that the pipeline works end to end, not as a measure of how it would do on real tickets.

**Next, notebook 8:** calibration. A decision model is only useful if its probabilities mean something, so we'll measure whether "90% confident" is right 90% of the time (expected calibration error) and fix it with temperature scaling, fitted on the validation set. After that: a real-text test using the public datasets, and LoRA.